# Benchmark analysis

Compare identical scenarios and report uncertainty without inventing repeated evidence.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Benchmark configuration

The simulator is deterministic. Repeating one scenario does not create independent observations.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt

from autonomous_vehicles.evaluation.statistics import bootstrap_mean, wilson_interval
from autonomous_vehicles.experiments.comparison import leaderboard
from autonomous_vehicles.experiments.config import SimulationConfig
from autonomous_vehicles.experiments.runner import run_simulation_suite

config = SimulationConfig(
    scenarios=("lane_following", "winding_road", "stopped_vehicle"),
    controllers=("pursuit", "stanley"),
)

## Run the suite

Each record identifies scenario, controller and repetition. The complete report records its environment.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    report = run_simulation_suite(config, Path(directory) / "suite")
print(report["aggregate"])
print(report["environment"])
print("Observation model:", report["observation_model"])
assert len(report["records"]) == 6

## Controller summary

An average across this hand-selected suite is descriptive and not a population claim.

In [ ]:
rows = leaderboard(report)
for row in rows:
    print(row)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([row["controller"] for row in rows], [row["mean"] for row in rows])
ax.set_ylabel("Mean lateral RMSE (m)")
ax.set_title("Selected synthetic scenarios")
plt.show()

## Events and progress

Collision and road departure are separate outcomes. A stopped vehicle may have low progress and no collision.

In [ ]:
for record in report["records"]:
    metric = record["metrics"]
    print(
        {
            "scenario": metric["scenario"],
            "controller": record["controller"],
            "collision": metric["collision"],
            "off_road": metric["off_road"],
            "progress_m": metric["progress_meters"],
            "termination": metric["termination"],
        }
    )

## Illustrate uncertainty

Bootstrap intervals here treat the selected scenarios as observations. Their interpretation is limited by suite selection.

In [ ]:
errors = [
    record["metrics"]["lateral_rmse_meters"]
    for record in report["records"]
    if record["controller"] == "pursuit"
]
estimate = bootstrap_mean(errors, resamples=200, seed=42)
collisions = sum(record["metrics"]["collision"] for record in report["records"])
interval = wilson_interval(collisions, len(report["records"]))
print("Descriptive error estimate:", estimate)
print("Illustrative collision interval:", interval)
print("Do not interpret a tiny deterministic suite as real-road evidence.")

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.